# 气象数据探索性分析 (EDA)

本笔记本旨在对 `st-missing-fill` 项目中的气象数据进行全面的统计与可视化分析。

**分析维度：**
1. **站点元数据分析**：站点的空间分布、海拔特征及类型统计。
2. **变量元数据分析**：气象参数的覆盖情况。
3. **时间序列与质量分析**：以温度数据为例，探索缺失值模式、相关性及时间变化趋势。

---

In [ ]:
# 1. 环境设置与依赖加载
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster

# 设置绘图风格 - 优雅美观
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] # 解决Mac中文显示问题
plt.rcParams['axes.unicode_minus'] = False

# 设置项目根目录
dir_path = '/Users/lxx/Documents/codes/st-missing-fill'
os.chdir(dir_path)
if dir_path not in sys.path:
    sys.path.append(dir_path)

print(f"工作目录已切换至: {os.getcwd()}")

## 2. 数据加载

加载站点信息、参数说明以及处理后的温度数据作为分析样本。

In [ ]:
# 加载元数据
stations = pd.read_parquet('data/meteo-swiss-description/stations.parquet')
params = pd.read_parquet('data/meteo-swiss-description/parameters.parquet')

# 加载温度数据样本 (2020-2025 处理后的数据)
temp_df = pd.read_parquet('data/processed-data/temperature.parquet')

print(f"站点数量: {len(stations)}")
print(f"参数数量: {len(params)}")
print(f"温度数据维度: {temp_df.shape}")
display(stations.head(3))
display(params.head(3))

## 3. 站点元数据分析

探索站点的地理特征（海拔、经纬度）和类型分布。

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 3.1 站点海拔分布
sns.histplot(data=stations, x='station_height', kde=True, color='skyblue', ax=ax[0])
ax[0].set_title('站点海拔分布直方图', fontsize=14, fontweight='bold')
ax[0].set_xlabel('海拔 (米)')
ax[0].set_ylabel('站点数量')

# 3.2 站点类型统计
type_counts = stations['station_type'].value_counts()
sns.barplot(x=type_counts.index, y=type_counts.values, palette='viridis', ax=ax[1])
ax[1].set_title('站点类型统计', fontsize=14, fontweight='bold')
ax[1].set_xlabel('站点类型')
ax[1].set_ylabel('数量')
for i, v in enumerate(type_counts.values):
    ax[1].text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 3.3 站点空间分布交互地图
m = folium.Map(
    location=[stations['latitude'].mean(), stations['longitude'].mean()],
    zoom_start=8,
    tiles='CartoDB positron'  # 使用简洁的底图
)

marker_cluster = MarkerCluster().add_to(m)

for idx, row in stations.iterrows():
    # 根据海拔设置颜色
    color = '#2ecc71' if row['station_height'] < 1000 else ('#f1c40f' if row['station_height'] < 2000 else '#e74c3c')
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        popup=f"<b>{row['station_name']}</b><br>海拔: {row['station_height']}m<br>类型: {row.get('station_type', 'N/A')}",
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7
    ).add_to(marker_cluster)

# 添加图例说明 (HTML)
legend_html = '''
     <div style="position: fixed; 
     bottom: 50px; left: 50px; width: 120px; height: 90px; 
     border:2px solid grey; z-index:9999; font-size:12px;
     background-color:white; opacity: 0.8; padding: 10px;">
     <b>海拔图例</b><br>
     <i style="background:#2ecc71;width:10px;height:10px;display:inline-block;border-radius:50%;"></i> < 1000m<br>
     <i style="background:#f1c40f;width:10px;height:10px;display:inline-block;border-radius:50%;"></i> 1000-2000m<br>
     <i style="background:#e74c3c;width:10px;height:10px;display:inline-block;border-radius:50%;"></i> > 2000m
      </div>
     '''
m.get_root().html.add_child(folium.Element(legend_html))

m

## 4. 时间序列与数据质量分析 (以温度数据为例)

分析数据的完整性，并探索时间变化的规律。

In [ ]:
# 4.1 缺失值热力图 (选取前50个站点和最近1个月的数据进行展示)
subset_df = temp_df.iloc[-4320:, :50]  # 最近30天 (10min频率 -> 6*24*30 = 4320)

plt.figure(figsize=(20, 8))
sns.heatmap(subset_df.isnull().T, cmap='magma', cbar=False, xticklabels=False)
plt.title('温度数据缺失值分布热力图 (最近30天, 前50个站点)', fontsize=16, fontweight='bold')
plt.xlabel('时间 (索引)')
plt.ylabel('站点')
plt.show()

In [ ]:
# 4.2 整体缺失率统计
missing_rates = temp_df.isnull().mean() * 100

plt.figure(figsize=(12, 5))
sns.histplot(missing_rates, bins=30, color='salmon', kde=True)
plt.title('各站点温度数据缺失率分布', fontsize=14, fontweight='bold')
plt.xlabel('缺失率 (%)')
plt.ylabel('站点数量')
plt.axvline(x=missing_rates.mean(), color='red', linestyle='--', label=f'平均缺失率: {missing_rates.mean():.2f}%')
plt.legend()
plt.show()

In [ ]:
# 4.3 站点间温度相关性分析 (选取完整度最高的前10个站点)
top_stations = missing_rates.sort_values().index[:10]
corr_matrix = temp_df[top_stations].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', 
            vmin=0.8, vmax=1.0, square=True, linewidths=.5)
plt.title('Top 10 完整站点温度相关性矩阵', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# 4.4 季节性与日变化分析 (选取第一个站点)
sample_station = temp_df.columns[0]
sample_series = temp_df[sample_station].dropna()

# 添加时间特征
df_plot = pd.DataFrame({'Temperature': sample_series})
df_plot['Hour'] = sample_series.index.hour
df_plot['Month'] = sample_series.index.month

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 日变化箱线图
sns.boxplot(data=df_plot, x='Hour', y='Temperature', palette='coolwarm', ax=ax[0])
ax[0].set_title(f'站点 {sample_station} 温度日变化规律', fontsize=12, fontweight='bold')
ax[0].set_xlabel('小时 (0-23)')
ax[0].set_ylabel('温度 (°C)')

# 月变化箱线图
sns.boxplot(data=df_plot, x='Month', y='Temperature', palette='Spectral', ax=ax[1])
ax[1].set_title(f'站点 {sample_station} 温度季节性变化规律', fontsize=12, fontweight='bold')
ax[1].set_xlabel('月份')
ax[1].set_ylabel('温度 (°C)')

plt.tight_layout()
plt.show()

## 5. 缺失模式分析

分析站点之间的距离关系，为空间缺失模式（SPATIAL）设计做准备。

In [ ]:
# 5.1 导入缺失模式生成模块
import sys
sys.path.append('..')
from src.missing_patterns import (
    MissingPatternGenerator, 
    calculate_station_distances,
    analyze_distance_distribution
)
from src.missing_generator import MissingDataGenerator

In [ ]:
# 5.2 计算站点间距离并分析分布
# 提取站点坐标 (纬度, 经度)
station_coords = stations[['latitude', 'longitude']].values

# 计算站点间距离矩阵
distance_matrix = calculate_station_distances(station_coords)

# 分析距离分布统计信息
dist_stats = analyze_distance_distribution(station_coords)

print("\n站点间距离统计信息:")
for key, value in dist_stats.items():
    print(f"  {key}: {value:.4f}")

In [ ]:
# 5.3 可视化站点间距离分布
# 提取上三角矩阵的距离值 (排除对角线)
import numpy as np
upper_triangle_idx = np.triu_indices_from(distance_matrix, k=1)
distances = distance_matrix[upper_triangle_idx]

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 距离分布直方图
ax[0].hist(distances, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax[0].axvline(x=dist_stats['median'], color='red', linestyle='--', linewidth=2, label=f"中位数: {dist_stats['median']:.2f}")
ax[0].axvline(x=dist_stats['q25'], color='orange', linestyle='--', linewidth=2, label=f"Q25: {dist_stats['q25']:.2f}")
ax[0].axvline(x=dist_stats['q75'], color='green', linestyle='--', linewidth=2, label=f"Q75: {dist_stats['q75']:.2f}")
ax[0].set_title('站点间距离分布直方图', fontsize=14, fontweight='bold')
ax[0].set_xlabel('距离 (度)')
ax[0].set_ylabel('频次')
ax[0].legend()
ax[0].grid(True, alpha=0.3)

# 距离热力图 (前20个站点)
subset_size = min(20, len(stations))
sns.heatmap(distance_matrix[:subset_size, :subset_size], 
            cmap='YlOrRd', annot=False, fmt='.2f',
            xticklabels=stations['station_name'].iloc[:subset_size],
            yticklabels=stations['station_name'].iloc[:subset_size],
            ax=ax[1], cbar_kws={'label': '距离 (度)'})
ax[1].set_title(f'站点距离矩阵热力图 (前{subset_size}个站点)', fontsize=14, fontweight='bold')
plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# 5.4 确定空间邻域阈值 z
# 建议使用 Q25 到中位数之间的值作为阈值
# 这样既能保证有足够的邻居，又不会影响太远的站点

suggested_threshold = (dist_stats['q25'] + dist_stats['median']) / 2
print(f"\n建议的空间邻域阈值 z: {suggested_threshold:.4f} 度")

# 计算在该阈值下每个站点的邻居数量
neighbors_count = (distance_matrix <= suggested_threshold).sum(axis=1) - 1  # 减1排除自己

plt.figure(figsize=(12, 5))
plt.bar(range(len(neighbors_count)), neighbors_count, color='teal', alpha=0.7)
plt.axhline(y=neighbors_count.mean(), color='red', linestyle='--', linewidth=2, 
            label=f'平均邻居数: {neighbors_count.mean():.1f}')
plt.title(f'各站点在阈值 z={suggested_threshold:.4f} 下的邻居数量', fontsize=14, fontweight='bold')
plt.xlabel('站点索引')
plt.ylabel('邻居数量')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n邻居数统计:")
print(f"  平均邻居数: {neighbors_count.mean():.2f}")
print(f"  最小邻居数: {neighbors_count.min()}")
print(f"  最大邻居数: {neighbors_count.max()}")

## 6. 缺失模式生成演示

演示三种缺失模式（MCAR、SEQ、SPATIAL）的生成效果。

In [ ]:
# 6.1 准备演示数据 (使用温度数据的一个子集)
# 选择最近7天的数据和前10个站点进行演示
demo_days = 7
demo_stations = 10
demo_timesteps = 144 * demo_days  # 10分钟频率，每天144个时间点

demo_data = temp_df.iloc[-demo_timesteps:, :demo_stations].copy()
demo_coords = station_coords[:demo_stations]

print(f"演示数据形状: {demo_data.shape}")
print(f"时间范围: {demo_data.index[0]} 到 {demo_data.index[-1]}")

In [ ]:
# 6.2 MCAR 模式演示
generator_mcar = MissingDataGenerator(demo_data, random_state=42)
data_mcar = generator_mcar.apply_mcar(missing_rate=0.3)

print("\nMCAR 模式统计:")
stats_mcar = generator_mcar.get_missing_statistics()
print(f"  总缺失率: {stats_mcar['missing_rate']:.2%}")
print(f"  总缺失值: {stats_mcar['missing_values']}")

# 可视化 MCAR 缺失模式
plt.figure(figsize=(16, 6))
sns.heatmap(data_mcar.isnull().T, cmap='RdYlGn_r', cbar=False, xticklabels=False)
plt.title('MCAR 缺失模式 (Missing Completely At Random)', fontsize=14, fontweight='bold')
plt.xlabel('时间')
plt.ylabel('站点')
plt.tight_layout()
plt.show()

In [ ]:
# 6.3 SEQ 模式演示
generator_seq = MissingDataGenerator(demo_data, random_state=42)
data_seq = generator_seq.apply_seq(missing_rate=0.3, max_length=100)

print("\nSEQ 模式统计:")
stats_seq = generator_seq.get_missing_statistics()
print(f"  总缺失率: {stats_seq['missing_rate']:.2%}")
print(f"  总缺失值: {stats_seq['missing_values']}")

# 可视化 SEQ 缺失模式
plt.figure(figsize=(16, 6))
sns.heatmap(data_seq.isnull().T, cmap='RdYlGn_r', cbar=False, xticklabels=False)
plt.title('SEQ 缺失模式 (Sequential Missing, max_length=100)', fontsize=14, fontweight='bold')
plt.xlabel('时间')
plt.ylabel('站点')
plt.tight_layout()
plt.show()

In [ ]:
# 6.4 SPATIAL 模式演示
generator_spatial = MissingDataGenerator(demo_data, random_state=42)
data_spatial = generator_spatial.apply_spatial(
    missing_rate=0.3,
    station_coords=demo_coords,
    distance_threshold=suggested_threshold,
    max_length=100,
    n_neighbors_range=(1, 3)
)

print("\nSPATIAL 模式统计:")
stats_spatial = generator_spatial.get_missing_statistics()
print(f"  总缺失率: {stats_spatial['missing_rate']:.2%}")
print(f"  总缺失值: {stats_spatial['missing_values']}")

# 可视化 SPATIAL 缺失模式
plt.figure(figsize=(16, 6))
sns.heatmap(data_spatial.isnull().T, cmap='RdYlGn_r', cbar=False, xticklabels=False)
plt.title('SPATIAL 缺失模式 (Spatially Correlated Sequential Missing)', fontsize=14, fontweight='bold')
plt.xlabel('时间')
plt.ylabel('站点')
plt.tight_layout()
plt.show()

In [ ]:
# 6.5 比较三种缺失模式
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# MCAR
sns.heatmap(data_mcar.isnull().T, cmap='RdYlGn_r', cbar=False, xticklabels=False, ax=axes[0])
axes[0].set_title(f'MCAR 模式 (缺失率: {stats_mcar["missing_rate"]:.2%})', fontsize=12, fontweight='bold')
axes[0].set_ylabel('站点')

# SEQ
sns.heatmap(data_seq.isnull().T, cmap='RdYlGn_r', cbar=False, xticklabels=False, ax=axes[1])
axes[1].set_title(f'SEQ 模式 (缺失率: {stats_seq["missing_rate"]:.2%}, max_length=100)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('站点')

# SPATIAL
sns.heatmap(data_spatial.isnull().T, cmap='RdYlGn_r', cbar=False, xticklabels=False, ax=axes[2])
axes[2].set_title(f'SPATIAL 模式 (缺失率: {stats_spatial["missing_rate"]:.2%}, threshold={suggested_threshold:.3f})', 
                  fontsize=12, fontweight='bold')
axes[2].set_ylabel('站点')
axes[2].set_xlabel('时间')

plt.tight_layout()
plt.show()

print("\n三种模式对比总结:")
print("1. MCAR: 完全随机缺失，缺失值在时间和空间上均匀分布")
print("2. SEQ: 序列缺失，在单个站点产生连续的缺失片段")
print("3. SPATIAL: 空间相关序列缺失，邻近站点同时产生连续缺失")